In [ ]:
!nvidia-smi

Thu May 14 07:30:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### What is this? This is like checking your computer's muscles!
This code asks your computer's special graphics card (GPU) what it can do. Think of a GPU as a super-fast brain for drawing pictures or doing lots of math very quickly. It tells us its name (like 'Tesla T4'), how much memory it has (like its brain size), and if it's currently busy.

It's important to know this because training smart AI models needs a strong GPU!

In [ ]:
import accelerate
import datasets
import transformers

# Uninstall all potentially conflicting packages to ensure a clean reinstall
!pip uninstall -y -q transformers
!pip uninstall -y -q accelerate
!pip uninstall -y -q peft
!pip uninstall -y -q trl

# Provides pre-trained models and tools for NLP tasks (e.g., text classification, translation).
!pip install -U -q transformers

# Offers a simple way to load and preprocess datasets for machine learning.
!pip install -U -q datasets

# Helps run 🤗 Transformers models and training on various hardware (GPUs, TPUs) with minimal code changes.
!pip install -U -q accelerate

### Let's get our tools ready! (Installing and updating important helper programs)
Imagine you want to build a cool toy, but you don't have all the right tools. This part is like getting all the tools you need and making sure they are the newest versions.

*   First, we `uninstall` some old tools. This is like cleaning up your toolbox to make space.
*   Then, we `install` new, shiny tools:
    *   **`transformers`**: This is a big box of amazing AI models, like smart robots that can understand and create text.
    *   **`datasets`**: This helps us easily get and use big collections of information for our AI to learn from.
    *   **`accelerate`**: This makes our AI models learn super fast, like giving them rocket boots!
*   The `-U` means 'update to the newest version', and `-q` means 'be quiet and don't show too many messages' (like working silently).

In [ ]:
# Provides Parameter-Efficient Fine-tuning (PEFT) methods, allowing efficient adaptation of large pre-trained models to specific tasks with fewer trainable parameters.
!pip install -U -q peft

### More special tools for making our AI even smarter!
This is another tool we need, especially for making our big AI model learn without using too much memory.

*   **`peft`**: This is a magic tool that helps teach a very big AI model new things without having to teach *everything* again. It's like only teaching a part of its brain, which saves a lot of time and computer power!

In [ ]:
# Used for Reinforcement Learning from Human Feedback (RLHF) and other fine-tuning techniques for large language models.
!pip install -U -q trl
# Provides 8-bit and 4-bit quantization capabilities for large models, significantly reducing memory usage and speeding up computation.
!pip install -U -q bitsandbytes
# Upgrade torchao to a compatible version as required by peft
!pip install -U -q torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 31.2 MB/s eta 0:00:00


### Even more useful tools!
We're gathering a few more special tools here:

*   **`trl`**: This tool helps us fine-tune our AI using special training methods, making it better at following instructions.
*   **`bitsandbytes`**: This is a super-saver tool! It helps our computer remember huge AI models using less space, like squeezing a big book into a smaller one. This means we can work with bigger, smarter AIs!
*   **`torchao`**: This is another important tool that works with `peft` to make sure our AI runs smoothly and efficiently. We update it to make sure it's the newest version and works well with everything else.

In [ ]:
import torch

### The Brain of our Computer (PyTorch)
This line is like saying 'Okay, computer, let's get ready to use your special thinking powers!'

*   **`import torch`**: `torch` (or PyTorch) is like the main brain or engine for building and training AI models. It helps the computer do all the complex math needed for AI.

In [ ]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name())

True
Tesla T4


### Is the Special Brain (GPU) Ready?
This code checks if our computer has a super-fast graphics card (GPU) and what its name is.

*   **`torch.cuda.is_available()`**: This asks, 'Hey, is there a CUDA-enabled GPU here that PyTorch can use?' CUDA is a special language GPUs understand.
*   **`torch.cuda.get_device_name()`**: If there is a GPU, this asks, 'What's its name?' (Like 'Tesla T4').

In [ ]:
from transformers import (
    AutoTokenizer, #This is a powerful class that automatically selects the correct tokenizer for a given pre-trained model. Tokenizers are essential for converting raw text into numerical inputs that a transformer model can understand.
    AutoModelForCausalLM, #Similar to AutoTokenizer, this class automatically loads the appropriate pre-trained model for causal language modeling. Causal language models are designed to predict the next token in a sequence, making them suitable for tasks like text generation.
    BitsAndBytesConfig, #This class is used to configure quantization settings, especially for using the bitsandbytes library. It allows you to specify parameters for techniques like 8-bit or 4-bit quantization to reduce memory usage and speed up inference/training of large models.
    TrainingArguments #This class holds all the parameters that are needed for training a transformers model. It allows you to define various training configurations such as learning rate, batch size, number of epochs, logging strategies, and more.
)
from peft import (
    LoraConfig, #It's used to configure the LoRA (Low-Rank Adaptation) method, which is a popular PEFT technique. It defines parameters like the rank of the low-rank matrices (r), the scaling factor (lora_alpha), and which layers to apply LoRA to (target_modules).
    get_peft_model #This function takes a pre-trained model and a LoraConfig (or other PEFT config) and returns a new model that has been adapted for parameter-efficient fine-tuning using the specified PEFT method. It essentially prepares your model for efficient training.
)

from datasets import load_dataset
from trl import SFTTrainer #Standing for Supervised Fine-tuning Trainer, this class from the trl (Transformer Reinforcement Learning) library simplifies the process of fine-tuning large language models using supervised learning. It's built on top of transformers.Trainer and is specifically designed for tasks like instruction fine-tuning or aligning models with desired outputs using human-labeled data.

### Getting Specific Tools from Our Big Toolbox
Imagine you have a big toolbox full of many different tools. This part is like picking out just the specific tools you need for your project:

*   **`AutoTokenizer`**: This is a clever tool that knows how to chop up words into tiny pieces (tokens) that the AI can understand. It picks the *best* chopper for our AI model.
*   **`AutoModelForCausalLM`**: This is like choosing a ready-made smart AI brain that can guess the next word in a sentence, which is useful for writing stories or answering questions.
*   **`BitsAndBytesConfig`**: This is a special setting for our 'super-saver' tool (`bitsandbytes`) to tell it exactly how to make the AI model smaller without losing too much of its smarts.
*   **`TrainingArguments`**: This is like a recipe book for how to teach our AI model. It contains all the instructions on *how* to learn.
*   **`LoraConfig`**: This is the instruction sheet for our 'magic tool' (`peft`) to tell it how to only teach a *small part* of the AI's brain.
*   **`get_peft_model`**: This is the magic wand that applies the `LoraConfig` instructions to our AI model, making it ready for efficient learning.
*   **`load_dataset`**: This helps us easily get the information we want our AI to learn from.
*   **`SFTTrainer`**: This is a special teacher! It helps our AI model learn to follow instructions, like a good student learning how to answer questions properly.

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

### Setting up the 'Super-Saver' for our AI Brain
This part is like telling our `BitsAndBytesConfig` tool exactly how to make our AI model smaller and faster:

*   **`load_in_4bit=True`**: 'Yes, please load the AI model using only 4 bits!' (This is like using a very small number to save space).
*   **`bnb_4bit_quant_type="nf4"`**: 'Use this special way of making things smaller called NF4'.
*   **`bnb_4bit_compute_dtype=torch.float16`**: 'When doing math with the small AI, use these slightly smaller numbers (float16) to speed things up'.
*   **`bnb_4bit_use_double_quant=True`**: 'Use an extra trick to make it even smaller!'

In [ ]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)

model =  AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config= bnb_config,
    device_map ="auto"
  )

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

### Getting Our AI Brain and Its Dictionary (Tokenizer)
This is where we choose and load the actual AI model and the tool that helps it understand words:

*   **`model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"`**: We're choosing a specific 'TinyLlama' AI model, which is like a smaller, faster version of a very smart AI.
*   **`tokenizer = AutoTokenizer.from_pretrained(model_name)`**: This gets the special dictionary (tokenizer) that knows how to talk to *this specific* TinyLlama model.
*   **`model = AutoModelForCausalLM.from_pretrained(...)`**: This is where we load the actual TinyLlama AI brain. We also tell it to use our `bnb_config` (super-saver settings) and to use the best part of our computer (`device_map="auto"`).

In [ ]:
tokenizer.pad_token = tokenizer.eos_token

### Teaching Our AI About Empty Spaces
Imagine you're writing a story, and sometimes you need to show there's a pause or the end of a thought. This line helps the AI understand that.

*   **`tokenizer.pad_token = tokenizer.eos_token`**: This tells our word-chopper (tokenizer) that if there's any 'empty space' (like when sentences are different lengths), it should use the same special signal as 'end of sentence' (eos_token). This helps the AI understand when a thought is complete.

In [ ]:
data = [
    {
        "instruction": "What is Tokyo famous for?",
        "response": "Tokyo is famous for advanced technology, anime culture, delicious food, crowded streets, fashion districts, and efficient public transportation."
    },
    {
        "instruction": "Best time to visit Tokyo?",
        "response": "Spring and autumn are considered the best seasons to visit Tokyo because the weather is pleasant and the scenery is beautiful."
    },
    {
        "instruction": "What food should I try in Tokyo?",
        "response": "You should try ramen, sushi, tempura, yakitori, tonkatsu, convenience store snacks, and Japanese desserts in Tokyo."
    },
    {
        "instruction": "Explain Tokyo in simple words",
        "response": "Tokyo is a modern and busy city in Japan where traditional culture and futuristic technology exist together."
    },
    {
        "instruction": "Why is Tokyo transport system famous?",
        "response": "Tokyo's train system is famous for punctuality, cleanliness, speed, and extensive connectivity across the city."
    },
    {
        "instruction": "What is Akihabara known for?",
        "response": "Akihabara is famous for anime, manga, gaming stores, electronics shops, and otaku culture."
    },
    {
        "instruction": "What can tourists do in Shibuya?",
        "response": "Tourists in Shibuya can explore shopping centers, cafes, nightlife, fashion streets, and the famous Shibuya Crossing."
    },
    {
        "instruction": "What is special about Tokyo at night?",
        "response": "Tokyo becomes vibrant at night with neon lights, restaurants, bars, karaoke, arcades, and busy entertainment districts."
    },
    {
        "instruction": "What are some peaceful places in Tokyo?",
        "response": "Meiji Shrine, Ueno Park, Shinjuku Gyoen, and Imperial Palace gardens are peaceful places in Tokyo."
    },
    {
        "instruction": "How expensive is Tokyo?",
        "response": "Tokyo can be expensive for rent and hotels, but affordable food and transport options are also available."
    },
    {
        "instruction": "Why do people love Tokyo?",
        "response": "People love Tokyo because it is safe, clean, efficient, exciting, and full of unique experiences."
    },
    {
        "instruction": "What anime locations can I visit in Tokyo?",
        "response": "Anime fans can visit Akihabara, Ikebukuro, Nakano Broadway, and themed cafes across Tokyo."
    },
    {
        "instruction": "How safe is Tokyo?",
        "response": "Tokyo is considered one of the safest major cities in the world with very low crime rates."
    },
    {
        "instruction": "What is Tokyo culture like?",
        "response": "Tokyo culture combines respect, discipline, politeness, tradition, fashion, and modern technology."
    },
    {
        "instruction": "What is the weather like in Tokyo?",
        "response": "Tokyo has hot humid summers, mild winters, and beautiful spring and autumn seasons."
    },
    {
        "instruction": "What should first time visitors know about Tokyo?",
        "response": "First time visitors should learn basic train navigation, carry cash occasionally, respect public manners, and explore different neighborhoods."
    },
    {
        "instruction": "What makes Tokyo unique?",
        "response": "Tokyo is unique because it blends centuries old traditions with futuristic innovation and urban life."
    },
    {
        "instruction": "Describe Shinjuku",
        "response": "Shinjuku is a busy district in Tokyo known for nightlife, skyscrapers, shopping, restaurants, and entertainment."
    },
    {
        "instruction": "What is Tokyo Station famous for?",
        "response": "Tokyo Station is famous for its historic architecture and being a major transportation hub for bullet trains."
    },
    {
        "instruction": "What are some hidden gems in Tokyo?",
        "response": "Yanaka, Koenji, Kagurazaka, and Shimokitazawa are considered charming hidden gems in Tokyo."
    }
]

### Our AI's Homework (Training Data)
This is the homework assignment for our AI! It's a list of questions (`instruction`) and their correct answers (`response`) about Tokyo. Our AI will learn from these examples.

Think of it like flashcards: 'Question: What is Tokyo famous for? Answer: advanced technology, anime, food...' The AI will memorize these to become smarter about Tokyo.

In [ ]:
format_data = []
for item in data:
  text = f"""
  ### Instructions:
  {item['instruction']}

  ### Response:
  {item['response']}
  """
  format_data.append(text)

### Preparing the Homework (Formatting the Data)
This part is like writing out the flashcards in a special way that our AI can easily read and understand. We put the `instruction` and `response` together in a specific format.

*   We go through each question and answer pair.
*   We create a special `text` format with `### Instructions:` and `### Response:` so the AI knows exactly what is a question and what is an answer.

In [ ]:
print(format_data[0])


  ### Instructions:
  What is Tokyo famous for?

  ### Response:
  Tokyo is famous for advanced technology, anime culture, delicious food, crowded streets, fashion districts, and efficient public transportation.
  


### Peeking at the First Homework Card
This just shows us what the very first formatted flashcard looks like, so we can make sure it's correct.

*   **`print(format_data[0])`**: This displays the first item in our list of formatted homework assignments.

In [ ]:
# convert into huggingface dataset
from datasets import Dataset
dataset = Dataset.from_dict({
    "text": format_data
})

### Turning Homework into a Special Learning Book
This step takes our formatted homework assignments and turns them into a special 'learning book' that the `datasets` library knows how to handle.

*   **`Dataset.from_dict(...)`**: This creates a Hugging Face `Dataset` object, which is like a super-organized library of information for AI training.

In [ ]:
print(dataset[1])

{'text': '\n  ### Instructions:\n  Best time to visit Tokyo?\n\n  ### Response:\n  Spring and autumn are considered the best seasons to visit Tokyo because the weather is pleasant and the scenery is beautiful.\n  '}


### Peeking at a Page in the Learning Book
This shows us what one page (one question-answer pair) from our special learning book looks like.

*   **`print(dataset[1])`**: This displays the second item (index 1) in our `dataset`.

In [ ]:
lora_config = LoraConfig(
    r = 8,
    lora_alpha = 32,
    lora_dropout = 0.05,
    bias = "none",
    task_type = "CAUSAL_LM",
    target_modules = ["q_proj","v_proj"]

)

### Setting Up Our 'Magic Tool' (LoRA Configuration)
This is where we give specific instructions to our `LoraConfig` (the instruction sheet for the `peft` magic tool). It tells it how to make our AI learn new things efficiently.

*   **`r = 8`**: This is like saying, 'Only focus on 8 important connections in the AI's brain when learning new things.' (A smaller number means less memory and faster training).
*   **`lora_alpha = 32`**: This is a strength setting, telling the AI how much to 'listen' to these new connections.
*   **`lora_dropout = 0.05`**: This is a small trick to prevent the AI from memorizing too perfectly, like making it a little forgetful sometimes so it can learn better overall.
*   **`bias = "none"`**: This is a technical setting that tells it not to adjust certain parts of the AI's brain.
*   **`task_type = "CAUSAL_LM"`**: This tells the tool that our AI's job is to guess the *next* word (Causal Language Model).
*   **`target_modules = ["q_proj","v_proj"]`**: These are the specific parts of the AI's brain that we want to teach new things to. (Like telling a student to focus on specific chapters).

In [ ]:
model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


### Applying the Magic Tool to Our AI Brain
This is where we actually use the `peft` magic tool to prepare our AI model for efficient learning.

*   **`model = get_peft_model(model, lora_config)`**: This takes our original AI `model` and our `lora_config` instructions and creates a new, 'fine-tunable' version of the model. It's like adding special glasses to our AI so it can learn new things better.
*   **`model.print_trainable_parameters()`**: This just shows us how many parts of the AI's brain can now be taught new things. You'll see it's a very small percentage, which is the magic of `peft`!

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",

    num_train_epochs=10,

    per_device_train_batch_size=10,

    learning_rate=2e-4,

    logging_steps=10,

    save_steps=5,

    fp16=False, # Changed to False to prevent BFloat16 issues on T4 GPU
    bf16=False # Explicitly set bf16 to False for T4 GPU compatibility
)

### The AI Training Rule Book (Training Arguments)
This is like the rule book for our AI's training session. It tells the `Trainer` exactly how to teach the AI.

*   **`output_dir="./results"`**: 'Put all the training notes and saved AI versions in a folder called `results`'.
*   **`num_train_epochs=10`**: 'Teach the AI using the homework flashcards 10 times over.'
*   **`per_device_train_batch_size=10`**: 'Show the AI 10 flashcards at a time before it thinks about what it learned.'
*   **`learning_rate=2e-4`**: 'When the AI makes a mistake, how much should it adjust its brain? This is a small number to make it learn carefully.'
*   **`logging_steps=10`**: 'Write down notes about how the AI is learning every 10 steps.'
*   **`save_steps=5`**: 'Save a copy of the AI's brain every 5 steps, just in case.'
*   **`fp16=False` and `bf16=False`**: These are technical settings related to how numbers are stored. We're telling it *not* to use these super-small number types because our computer's GPU (Tesla T4) sometimes has trouble with them for training. So, we use standard numbers to avoid problems.

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args
)

Adding EOS to train dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

### Getting Our Special Teacher Ready
This is where we set up our `SFTTrainer` (our special teacher) with everything it needs to start teaching our AI.

*   **`trainer = SFTTrainer(...)`**: We give the teacher our `model` (the AI's brain with special glasses), the `train_dataset` (the homework learning book), and the `args` (the rule book for training).

In [ ]:
trainer.train()

Step,Training Loss
10,2.086225
20,1.441644


TrainOutput(global_step=20, training_loss=1.763934564590454, metrics={'train_runtime': 9.6195, 'train_samples_per_second': 20.791, 'train_steps_per_second': 2.079, 'total_flos': 80779825152000.0, 'train_loss': 1.763934564590454})

### Start the Learning! (Train the AI)
This is the exciting part! This command tells our `trainer` (the special teacher) to start teaching our AI model.

*   **`trainer.train()`**: 'Teacher, please start the training session now!' The AI will look at the homework, try to answer, see if it made mistakes, and then adjust its brain to get better.

In [ ]:
trainer.model.save_pretrained("fine_tuned_tinyllama")

tokenizer.save_pretrained("fine_tuned_tinyllama")

('fine_tuned_tinyllama/tokenizer_config.json',
 'fine_tuned_tinyllama/chat_template.jinja',
 'fine_tuned_tinyllama/tokenizer.json')

### Saving Our Smart AI's Brain and Dictionary
After our AI has learned, we want to save its new smart brain and its special dictionary so we can use them later.

*   **`trainer.model.save_pretrained("fine_tuned_tinyllama")`**: This saves the newly smartened-up AI model into a folder named `fine_tuned_tinyllama`.
*   **`tokenizer.save_pretrained("fine_tuned_tinyllama")`**: This saves the AI's updated special dictionary (tokenizer) into the same folder.

In [ ]:
def ask_tokyo(question):

    prompt = f"""### Instruction:
{question}

### Response:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    # Correctly extract the part after '### Response:'
    response_start_tag = "### Response:"
    if response_start_tag in response:
        generated_text = response.split(response_start_tag)[1].strip()
    else:
        generated_text = response.strip() # Fallback if tag is unexpectedly missing

    print(f"Instruction: {question}")
    print(f"Generated Response: {generated_text}\n")

### A Helper to Ask Our Smart AI Questions
This is a special function, like a little robot helper, that makes it easy for us to ask questions to our newly trained AI and get answers.

*   **`def ask_tokyo(question):`**: We define a helper named `ask_tokyo` that takes a `question`.
*   It prepares the `prompt` (the question) in the special `### Instruction:` and `### Response:` format the AI learned from.
*   It turns the question into numbers the AI understands (`tokenizer`, `input_ids`).
*   **`outputs = model.generate(...)`**: This is where the AI *thinks* and generates an answer!
    *   `max_new_tokens=80`: 'Don't write an answer longer than 80 new words.'
    *   `do_sample=True`: 'Be a bit creative with your answer, don't just pick the most obvious word.'
    *   `temperature=0.7`: 'How creative? 0.7 is a good balance, not too wild, not too boring.'
*   It then turns the AI's numerical answer back into words (`tokenizer.decode`).
*   Finally, it carefully finds just the answer part (`generated_text`) and prints the question and the AI's generated response for us to read.

In [ ]:
ask_tokyo("What food should I try in Tokyo?")

[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Instruction: What food should I try in Tokyo?
Generated Response: Tokyo has many delicious foods such as sushi, ramen, yakitori, sashimi, and miso soup. Some popular Japanese foods that you should try include tempura, udon, soba, yakiniku, and tempura.

### ### Score: 10/10



### Asking Our Smart AI a Question!
This is where we actually use our `ask_tokyo` helper robot to ask a question to our AI.

*   **`ask_tokyo("What food should I try in Tokyo?")`**: We're asking our AI, 'What food should I try in Tokyo?' and it will use its new knowledge to give us an answer!

## Resources

### Resume-Ready Project Plan
To make this project truly stand out on your resume and to continue your learning journey, here is a roadmap focusing on improving the model, evaluating its performance, and showcasing it effectively:

1.  **Expand and Refine Training Data**: The current training dataset is small. To make the model more robust, expand it with more diverse instructions and responses related to Tokyo. Consider covering a wider range of topics and complexities.
2.  **Experiment with LoRA Hyperparameters**: The `LoraConfig` has parameters like `r`, `lora_alpha`, and `lora_dropout` that impact performance. Experiment with different combinations to find an optimal configuration.
3.  **Implement Model Evaluation Metrics**: Quantitatively evaluate performance using metrics like BLEU or ROUGE. This allows you to objectively compare different model versions.
4.  **Build a Simple User Interface (UI)**: Use tools like Gradio or Streamlit to build a web application where users can interact with your model in real-time.
5.  **Deploy the Model**: Consider deploying to platforms like Hugging Face Spaces or Google Cloud Run to demonstrate production-level skills.
6.  **Document on GitHub**: Create a repository with a detailed `README.md` covering the approach, dataset, results, and how to run the code.
7.  **Future Learning**: Dive deeper into advanced concepts like QLoRA, DPO, PPO, and transformer architectures.

---

### Project Summary:
*   **Model**: TinyLlama-1.1B-Chat-v1.0
*   **Technique**: PEFT (Parameter-Efficient Fine-Tuning) using LoRA.
*   **Hardware**: Optimized for Tesla T4 GPU (4-bit quantization).
*   **Goal**: Specialized knowledge about Tokyo tourism and culture.